<a href="https://colab.research.google.com/github/TOTVScontext/DataScience/blob/main/Context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

!pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 73.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
#Grupo 1 standad library
import re
from pathlib import Path
from collections import Counter
from typing import Optional

#Grupo 2 Third Party
import pandas as pd
import spacy
import nltk
from transformers import pipeline #Ia para achar os fragmentos das reunioes sobre preco e incertezas

#Grupo 3 visualizacao e interface
import plotly.express as px
from IPython.display import display
!python -m spacy download pt_core_news_sm
nltk.download('stopwords')

In [3]:
BASE_DIR = Path.cwd()
CAMINHO_DADOS = BASE_DIR / "ANON_nome_transcricao.csv"

STOP_WORDS_PT: set[str] = set(nltk.corpus.stopwords.words('portuguese'))

#nlp
STOP_WORDS_CUSTOM: set[str] = {
    'pra', 'aí', 'então', 'né', 'tá', 'gente', 'aqui', 'vai', 'fazer',
    'ter', 'porque', 'lá', 'assim', 'ser', 'vou', 'agora', 'tudo',
    'bem', 'entendi', 'falou', 'acho', 'vamos', 'pode', 'sobre', 'dá',
    'voc', 'você', 'n', 't', 'pro', 'mim', 'coisa', 'parte', 'dia', 'exemplo',
    'cara'
}

STOP_WORDS_PT.update(STOP_WORDS_CUSTOM)

#regex - n-grams
CATEGORIAS: dict[str, str] = {
    'Financeiro': 'contas pagar|pagar receber|financeiro|boleto|faturamento|nota|fical|pix|cartão|transferência|valor',
    'RH/Operacional': 'relógio ponto|ponto|jornada|colaborador|horas',
    'Vendas/Produtividade': 'vendedor|otimizar|venda|proposta|pipeline|crm',
    'Atendimento': 'falar|cliente|atendimento|suporte|contato'
}

In [ ]:
if CAMINHO_DADOS.is_file():
    print(f"Arquivo encontrado em: {CAMINHO_DADOS}")
    df = pd.read_csv(CAMINHO_DADOS, sep=',', on_bad_lines='skip',
                     encoding='utf-8', encoding_errors='ignore')
    print(f"Sucesso! Dataset carregado com {len(df)} linhas.")
else:
    raise FileNotFoundError(f" Erro Crítico: O arquivo '{CAMINHO_DADOS}' não foi localizado. Verifique se o nome ou a pasta estão corretos.")

In [ ]:
#LIMPEZA E EXTRAÇÃO
nlp = spacy.load("pt_core_news_sm")

def limpar_transcricao(texto: Optional[str]) -> str:
    if pd.isna(texto):
        return ""
    texto = re.sub(r'\[.*?\]', '', texto)
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúãõçâêôà\s]', ' ', texto)
    texto = ' '.join(texto.split())
    return texto

def categorizar_reuniao(texto: str) -> list[str]:
  tags_encontradas = []
  tags_encontradas = [tema for tema, pattern in CATEGORIAS.items()
                        if re.search(pattern, texto, re.IGNORECASE)]
  return tags_encontradas if tags_encontradas else ['Geral']

#tecnica de zero-shot(isolado para nao carregar toda hora)
classificador = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")


def extrair_palavras_chave_simples(texto: str) -> list[str]:
    palavras = texto.split()
    return [p for p in palavras if p not in STOP_WORDS_PT and len(p) > 2]

#para substantivo
def extrair_palavras_chave_nlp(texto: str) -> list[str]:
    doc = nlp(texto)
    return [
        token.text for token in doc
        if token.text not in STOP_WORDS_PT
        and len(token.text) > 2
        and token.pos_ == "NOUN"
    ]

def identificar_objecoes_ia(texto: str) -> str:
    if len(texto) < 10: return "Neutro"
    labels_objetivo = ["problema de orçamento ou preço", "dificuldade técnica ou dúvida", "menção a empresa concorrente", "satisfação e elogio"]
    resultado = classificador(texto, labels_objetivo, hypothesis_template="Este trecho indica {}.")

    top_label = resultado['labels'][0]
    top_score = resultado['scores'][0]
    return top_label if top_score > 0.4 else "Assunto Geral"


In [ ]:
# Limpeza e Gatilhos
print("Limpando transcrições e gerando gatilhos.")
df['TEXTO_LIMPO'] = df['ANON_TRANSCRICAO'].apply(limpar_transcricao)

gatilhos = 'whatsapp|ligar|contato|agendar|retorno|proxima|numero|reuniao'
df['TEM_PROXIMO_PASSO'] = df['TEXTO_LIMPO'].str.contains(gatilhos, case=False, na=False)

print(f"Processamento concluído. {df['TEM_PROXIMO_PASSO'].sum()} gatilhos encontrados.")
print("--- CONTADOR DE GATILHOS ---")
print(df['TEM_PROXIMO_PASSO'].value_counts())

# 2. NLP (Tags e Tokens)
print("\n Processando Tags e extraindo substantivos (Spacy).")
df['TOKENS'] = df['TEXTO_LIMPO'].apply(extrair_palavras_chave_nlp)
df['TAGS'] = df['TEXTO_LIMPO'].apply(categorizar_reuniao)

# 3. Teste de Ouro e IA
df_ouro = df[(df['TEM_PROXIMO_PASSO'] == True) & (df['TEXTO_LIMPO'].str.len() > 20)].copy()
print(f"\nFiltragem concluída! Das 87 reuniões, temos {len(df_ouro)} prontas para a análise profunda.")

print("Iniciando a IA nas reuniões de elite... aguarde.")
df_ouro['IA_OBJECOES'] = df_ouro['TEXTO_LIMPO'].apply(identificar_objecoes_ia)

print("\n--- AMOSTRA DOS 10 PRIMEIROS GATILHOS ENCONTRADOS ---")
display(df[df['TEM_PROXIMO_PASSO'] == True][['TEXTO_LIMPO']].head(10))

print("\n--- RESULTADO DO TESTE DE OURO ---")
display(df_ouro[['TEXTO_LIMPO', 'IA_OBJECOES']].head(15))





In [ ]:
# ---  Gráfico de Temas (Tags) ---
df_tags_contagem = df.explode('TAGS')
df_interessante = df_tags_contagem[df_tags_contagem['TAGS'] != 'Geral']

fig_temas = px.bar(df_interessante['TAGS'].value_counts().reset_index(),
                   x='count', y='TAGS', orientation='h',
                   title='Pilar 1 (B): Distribuição de Temas (Tags)',
                   color='count', color_continuous_scale='Viridis')
fig_temas.show()

# --- Gráfico de Objeções (IA) ---
df_final_viz = df_ouro[~df_ouro['IA_OBJECOES'].isin(['Assunto Geral', 'Neutro'])]

fig_objecoes = px.pie(df_final_viz, names='IA_OBJECOES',
             title='Pilar 1 (C): Análise Semântica de Objeções (IA Deep Learning)',
             hole=0.4,
             color_discrete_sequence=px.colors.qualitative.Pastel)
fig_objecoes.update_traces(textinfo='percent+label')
fig_objecoes.show()

print(f"Total de reuniões analisadas pela IA: {len(df_ouro)}")
print(f"Objeções claras detectadas: {len(df_final_viz)}\n")

# --- Gráfico de Termos Mais Frequentes ---
todas_palavras = [palavra for sublist in df['TOKENS'] for palavra in sublist]
contagem = Counter(todas_palavras)

top_palavras = pd.DataFrame(contagem.most_common(20), columns=['Palavra', 'Frequência'])

fig_palavras = px.bar(top_palavras, x='Frequência', y='Palavra',
                      orientation='h',
                      title='Top 20 Termos Reais de Negócios (Dados Limpos)',
                      color='Frequência',
                      color_continuous_scale=px.colors.sequential.Teal)
fig_palavras.update_layout(yaxis={'categoryorder':'total ascending'})
fig_palavras.show()

In [ ]:
#celula de inspecao final

# Vamos ver o que tem nas 10 primeiras linhas de 'TEXTO_LIMPO' e da original
display(df[['ANON_TRANSCRICAO', 'TEXTO_LIMPO']].head(10))

In [ ]:
#Codigo gerado por ia para achar padroes e identificar as categorias principais das reunioes (UTILIZADO SOMENTE PARA FINS DE PRODUTIVIDADE)
'''from sklearn.feature_extraction.text import CountVectorizer

docs = df['TEXTO_LIMPO'].dropna().head(10000)
vec = CountVectorizer(ngram_range=(2, 2), stop_words=list(stop_words_pt)).fit(docs)
bag_of_words = vec.transform(docs)
sum_words = bag_of_words.sum(axis=0)

words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)

print("Os 15 assuntos (pares) mais falados são:")
display(words_freq[:15])'''


#TESTE COM A IA
'''df_real = df.dropna(subset=['ANON_TRANSCRICAO']).copy()
amostra_viva = df_real.sample(5)

print("Agora sim! Analisando textos que existem...")
amostra_viva['IA_OBJECOES'] = amostra_viva['TEXTO_LIMPO'].apply(identificar_objecoes_ia)

display(amostra_viva[['TEXTO_LIMPO', 'IA_OBJECOES']])'''